# ResearchScope: write paper fields with Gemma 4 (backfill)

Runs `scripts/llm_enrich/enrich.py` over every paper in
[researchscope-papers](https://huggingface.co/datasets/kishormorol/researchscope-papers) with
`gemma-4-12b-it` on vLLM. **Needs an A100 or L4 runtime** (Runtime → Change runtime type): on a
T4, vLLM's attention kernel for Gemma 4 does not fit in shared memory.

Output goes to Google Drive and the run is resumable: if Colab disconnects, run all cells again
and it continues where it stopped. When it finishes, the last cell prints the file to hand back
(`llm_fields.jsonl.gz`).

Kaggle credentials are read from Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the
left sidebar) to download the model from Kaggle.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/researchscope-llm"
!mkdir -p {OUT_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U vllm kagglehub huggingface_hub
!rm -rf /content/ResearchScope && git clone -q --depth 1 https://github.com/kishormorol/ResearchScope /content/ResearchScope

In [ ]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
from huggingface_hub import hf_hub_download
MODEL = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b-it")
PAPERS = hf_hub_download("kishormorol/researchscope-papers", "data/papers.jsonl", repo_type="dataset")
print(MODEL, PAPERS)

In [ ]:
!cd /content/ResearchScope && python scripts/llm_enrich/enrich.py \
    --papers {PAPERS} --out {OUT_DIR}/llm_fields.jsonl \
    --model {MODEL} --model-name gemma-4-12b-it --batch 1024

In [ ]:
import gzip, json, shutil, collections
rows = [json.loads(l) for l in open(f"{OUT_DIR}/llm_fields.jsonl")]
flags = collections.Counter(f.split(":")[0] for r in rows for f in r["flags"])
print(f"{len(rows):,} papers, {sum(not r['flags'] for r in rows):,} clean; flags: {dict(flags)}")
with open(f"{OUT_DIR}/llm_fields.jsonl", "rb") as src, gzip.open(f"{OUT_DIR}/llm_fields.jsonl.gz", "wb") as dst:
    shutil.copyfileobj(src, dst)
print("Hand back:", f"{OUT_DIR}/llm_fields.jsonl.gz")